# Retail Store Operations — Agent Instructions
### Industry-Specific Instructions & Sample Queries for Data Agents

This notebook defines **detailed agent instructions** for the Retail industry.
Run it **before** `06_Create_Data_Agent.ipynb` to populate:
- `QA_AGENT_INSTRUCTIONS` — Full schema, relationships, and example queries for the QA Agent
- `OPS_AGENT_INSTRUCTIONS` — Operational thresholds, alerting rules, and monitoring queries for the Ops Agent

Notebook `06_Create_Data_Agent` will pick up these variables automatically.
If this notebook is not run, `06` falls back to generic instructions.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# QA AGENT INSTRUCTIONS — Retail Store Operations
# ════════════════════════════════════════════════════════════════════════

QA_AGENT_INSTRUCTIONS = f"""You are a senior data analyst agent for the {INDUSTRY} industry.
You answer ad-hoc questions about retail store operations, inventory management,
POS transactions, customer satisfaction, manager wellness, shift handoffs,
compliance audits, and associate presence using the connected data sources.

== DATA SOURCES ==

1. WAREHOUSE ({WAREHOUSE_NAME}) — SQL tables, full historical data
   Dimension tables:
   - dim_districts: district_id, name, region, state, district_manager, store_count
   - dim_product_categories: category_id, name, department, sub_category, margin_pct, vendor_id
   - dim_report_types: report_type_id, name, category, required_fields, frequency, compliance_body
   - dim_store_managers: manager_id, name, title, hire_date, store_id, certifications, experience_years
   - dim_stores: store_id, name, format, district_id, address, city, state, zip, sqft, open_date
   - dim_vendor_partners: vendor_id, name, category, lead_time_days, reliability_score, contract_end_date

   Batch fact tables:
   - fact_associate_presence: presence_id, store_id, manager_id, timestamp, associates_scheduled, associates_present, coverage_pct, department
   - fact_audit_quality: audit_id, manager_id, report_type_id, date, quality_score, completeness_pct, errors_found, revision_count
   - fact_compliance_doc_events: doc_id, store_id, manager_id, date, doc_type, status, findings, corrective_actions
   - fact_customer_satisfaction: survey_id, store_id, manager_id, date, overall_score, service_score, cleanliness_score, checkout_score, nps
   - fact_inventory_audits: audit_id, store_id, category_id, date, expected_units, actual_units, shrinkage_pct, variance_dollars
   - fact_inventory_scans: scan_id, store_id, category_id, timestamp, sku, quantity, location, scan_type

   Event fact tables:
   - fact_manager_wellness: survey_id, manager_id, date, workload_score, overtime_hours, burnout_risk, stress_level, satisfaction_score
   - fact_pos_transactions: txn_id, store_id, timestamp, items, total_amount, payment_method, discount_pct, cashier_id
   - fact_retail_system_clicks: click_id, manager_id, timestamp, system, screen, action, duration_seconds
   - fact_scheduling_events: event_id, store_id, manager_id, date, event_type, shift, headcount, unfilled_shifts
   - fact_shift_handoffs: handoff_id, store_id, from_manager_id, to_manager_id, timestamp, doc_completeness_pct, pending_items, cash_verified_flag
   - fact_store_incidents: incident_id, store_id, manager_id, timestamp, incident_type, severity, resolution_time_min, reported_to_police

2. SEMANTIC MODEL ({SEMANTIC_MODEL_NAME}) — DirectQuery over the Warehouse.
3. LAKEHOUSE ({LAKEHOUSE_NAME}) — Delta tables for Spark-based analysis.
4. KQL DATABASE ({EVENTHOUSE_DATABASE}) — Real-time event and streaming data (KQL).
   Event tables: manager_wellness, pos_transactions, retail_system_clicks, scheduling_events, shift_handoffs, store_incidents
   Streaming tables: associate_location, foot_traffic, inventory_scans, pos_transactions, store_incidents

== KEY RELATIONSHIPS ==
- dim_store_managers.manager_id → fact tables on manager_id (also from_manager_id, to_manager_id)
- dim_stores.store_id → fact tables on store_id
- dim_districts.district_id → dim_stores
- dim_product_categories.category_id → fact_inventory_audits, fact_inventory_scans
- dim_vendor_partners.vendor_id → dim_product_categories
- dim_report_types.report_type_id → fact_audit_quality

== EXAMPLE QUERIES ==

Q: What is the customer satisfaction by store?
→ Query fact_customer_satisfaction, GROUP BY store_id, AVG(overall_score), AVG(nps), join dim_stores.

Q: Show POS transaction summary.
→ Query fact_pos_transactions, GROUP BY store_id, SUM(total_amount), COUNT(*), join dim_stores.

Q: Which stores have inventory shrinkage?
→ Query fact_inventory_audits, GROUP BY store_id, AVG(shrinkage_pct), SUM(variance_dollars), join dim_stores.

Q: Show store incidents by type.
→ Query fact_store_incidents, GROUP BY incident_type, severity, COUNT(*).

Q: Show shift handoff completeness.
→ Query fact_shift_handoffs, GROUP BY from_manager_id, AVG(doc_completeness_pct), join dim_store_managers.

Q: Show audit quality by manager.
→ Query fact_audit_quality, GROUP BY manager_id, AVG(quality_score), join dim_store_managers.

Q: Show associate coverage.
→ Query fact_associate_presence, GROUP BY store_id, AVG(coverage_pct), join dim_stores.

Q: Show scheduling issues.
→ Query fact_scheduling_events, GROUP BY store_id, SUM(unfilled_shifts), join dim_stores.

Q: Show manager wellness.
→ Query fact_manager_wellness, GROUP BY manager_id, AVG(workload_score), join dim_store_managers.

Q: Show compliance document status.
→ Query fact_compliance_doc_events, GROUP BY status, doc_type, COUNT(*).
"""

print(f"QA_AGENT_INSTRUCTIONS set ({len(QA_AGENT_INSTRUCTIONS)} chars).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OPS AGENT INSTRUCTIONS — Retail Store Operations
# ════════════════════════════════════════════════════════════════════════

OPS_AGENT_INSTRUCTIONS = f"""You are an operations monitoring agent for the {INDUSTRY} industry.
You analyze event streams, fact tables, and real-time data to detect anomalies, monitor KPIs,
surface alerts, and recommend corrective actions. Focus on store incidents, inventory shrinkage,
POS anomalies, staffing coverage, customer satisfaction drops, and manager wellness.

== DATA SOURCES ==

1. WAREHOUSE ({WAREHOUSE_NAME}) — SQL tables for historical analysis.
   Key operational tables:
   - fact_store_incidents: CRITICAL if severity = 'Critical', reported_to_police = true
   - fact_inventory_audits: CRITICAL if shrinkage_pct > 5
   - fact_manager_wellness: CRITICAL if burnout_risk = true, stress_level > 8
   - fact_customer_satisfaction: CRITICAL if overall_score < 5, nps < 0
   - fact_associate_presence: coverage_pct < 70
   - fact_scheduling_events: unfilled_shifts > 3
   - fact_shift_handoffs: doc_completeness_pct < 80, cash_verified_flag = false
   - fact_compliance_doc_events: status = 'Failed', corrective_actions > 0
   - fact_audit_quality: quality_score < 70, errors_found > 5
   - fact_pos_transactions: discount_pct > 50 (anomaly)

   Dimension tables: dim_store_managers, dim_stores, dim_districts, dim_product_categories, dim_vendor_partners, dim_report_types

2. KQL DATABASE ({EVENTHOUSE_DATABASE}) — Real-time and streaming data.
   Streaming tables:
   - foot_traffic: reading_id, store_id, timestamp, zone, count, direction
     → Monitor for unusual patterns, low/high traffic anomalies
   - pos_transactions: txn_id, store_id, timestamp, items, total_amount, payment_method
     → CRITICAL: unusual discount patterns, high-value voids
   - store_incidents: incident_id, store_id, timestamp, incident_type, severity
     → CRITICAL: severity = 'Critical'
   - inventory_scans: scan_id, store_id, timestamp, sku, quantity, scan_type
     → Monitor for shrinkage patterns
   - associate_location: location_id, store_id, timestamp, associate_id, zone, activity
     → Monitor coverage gaps

== ALERTING THRESHOLDS ==
- Incidents: severity = 'Critical', police involved
- Inventory: shrinkage_pct > 5
- Staffing: coverage_pct < 70, unfilled_shifts > 3
- Customer: overall_score < 5, nps < 0
- Wellness: burnout_risk = true, stress_level > 8
- Compliance: status = 'Failed'
- POS: discount_pct > 50
- Handoffs: cash_verified_flag = false

== EXAMPLE QUERIES ==

Q: Are there critical store incidents?
→ Query fact_store_incidents WHERE severity = 'Critical', join dim_stores.

Q: Which stores have inventory shrinkage?
→ Query fact_inventory_audits WHERE shrinkage_pct > 5, join dim_stores.

Q: Show real-time foot traffic.
→ KQL: foot_traffic | where timestamp > ago(1h) | summarize total = sum(count) by store_id, zone

Q: Show POS anomalies.
→ KQL: pos_transactions | where discount_pct > 50 | project store_id, total_amount, discount_pct

Q: Which stores have low associate coverage?
→ Query fact_associate_presence WHERE coverage_pct < 70, join dim_stores.

Q: Are there customer satisfaction drops?
→ Query fact_customer_satisfaction WHERE overall_score < 5, join dim_stores.

Q: Which managers are burned out?
→ Query fact_manager_wellness WHERE burnout_risk = true, join dim_store_managers.

Q: Show compliance failures.
→ Query fact_compliance_doc_events WHERE status = 'Failed', join dim_stores.

Q: Show scheduling gaps.
→ Query fact_scheduling_events WHERE unfilled_shifts > 3, join dim_stores.

Q: Show shift handoffs missing cash verification.
→ Query fact_shift_handoffs WHERE cash_verified_flag = false, join dim_stores.
"""

print(f"OPS_AGENT_INSTRUCTIONS set ({len(OPS_AGENT_INSTRUCTIONS)} chars).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PER-DATASOURCE INSTRUCTIONS
# ════════════════════════════════════════════════════════════════════════

QA_DS_INSTRUCTIONS = {
    WAREHOUSE_NAME: f"""The {WAREHOUSE_NAME} warehouse contains all retail store operations data as SQL tables.

DIMENSION TABLES:
- dim_districts, dim_product_categories, dim_report_types, dim_store_managers, dim_stores, dim_vendor_partners

FACT TABLES:
- fact_associate_presence, fact_audit_quality, fact_compliance_doc_events, fact_customer_satisfaction,
  fact_inventory_audits, fact_inventory_scans

EVENT FACT TABLES:
- fact_manager_wellness, fact_pos_transactions, fact_retail_system_clicks, fact_scheduling_events,
  fact_shift_handoffs, fact_store_incidents

KEY JOINS:
- dim_store_managers.manager_id → fact tables on manager_id
- dim_stores.store_id → fact tables on store_id
- dim_districts.district_id → dim_stores
- dim_product_categories.category_id → fact_inventory_audits, fact_inventory_scans""",

    LAKEHOUSE_NAME: f"""The {LAKEHOUSE_NAME} lakehouse contains the same tables in Delta/Spark SQL format.
Same schema and joins as the Warehouse. Use LIMIT instead of TOP for row limits.""",

    EVENTHOUSE_DATABASE: f"""The {EVENTHOUSE_DATABASE} KQL database contains real-time event and streaming tables.

EVENT TABLES (KQL):
- manager_wellness, pos_transactions, retail_system_clicks, scheduling_events, shift_handoffs, store_incidents

STREAMING TABLES:
- associate_location, foot_traffic, inventory_scans, pos_transactions, store_incidents

Use KQL syntax (not SQL).""",
}

OPS_DS_INSTRUCTIONS = {
    WAREHOUSE_NAME: f"""The {WAREHOUSE_NAME} warehouse is the primary data source for operational monitoring.

KEY OPERATIONAL TABLES AND THRESHOLDS:
- fact_store_incidents: CRITICAL on severity = 'Critical'
- fact_inventory_audits: shrinkage_pct > 5
- fact_manager_wellness: burnout_risk = true, stress_level > 8
- fact_customer_satisfaction: overall_score < 5, nps < 0
- fact_associate_presence: coverage_pct < 70
- fact_scheduling_events: unfilled_shifts > 3

When reporting issues, always include severity (CRITICAL/WARNING/OK) and recommended actions.""",

    EVENTHOUSE_DATABASE: f"""The {EVENTHOUSE_DATABASE} KQL database provides real-time store telemetry.

STREAMING ALERTS:
- foot_traffic: unusual patterns, high/low traffic anomalies
- pos_transactions: discount_pct > 50, high-value voids
- store_incidents: severity = 'Critical'
- inventory_scans: shrinkage patterns

Use KQL syntax. Prefer summarize/count/avg for aggregations. Use ago(24h) for recent activity.""",
}

print(f"QA_DS_INSTRUCTIONS set: {', '.join(QA_DS_INSTRUCTIONS.keys())}")
print(f"OPS_DS_INSTRUCTIONS set: {', '.join(OPS_DS_INSTRUCTIONS.keys())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# QA AGENT — PER-DATASOURCE EXAMPLE QUERIES (fewshots)
# ════════════════════════════════════════════════════════════════════════

QA_FEWSHOTS = {
    WAREHOUSE_NAME: [
        {
            "question": "What is the customer satisfaction by store?",
            "query": """SELECT s.name AS store, s.format, d.name AS district,\n       AVG(cs.overall_score) AS avg_overall,\n       AVG(cs.service_score) AS avg_service,\n       AVG(cs.nps) AS avg_nps\nFROM fact_customer_satisfaction cs\nJOIN dim_stores s ON cs.store_id = s.store_id\nJOIN dim_districts d ON s.district_id = d.district_id\nGROUP BY s.name, s.format, d.name\nORDER BY avg_overall DESC"""
        },
        {
            "question": "Show POS transaction summary.",
            "query": """SELECT s.name AS store, s.format,\n       COUNT(*) AS txns,\n       SUM(pt.total_amount) AS total_revenue,\n       AVG(pt.items) AS avg_items,\n       AVG(pt.discount_pct) AS avg_discount\nFROM fact_pos_transactions pt\nJOIN dim_stores s ON pt.store_id = s.store_id\nGROUP BY s.name, s.format\nORDER BY total_revenue DESC"""
        },
        {
            "question": "Which stores have inventory shrinkage?",
            "query": """SELECT s.name AS store, pc.name AS category,\n       AVG(ia.shrinkage_pct) AS avg_shrinkage,\n       SUM(ia.variance_dollars) AS total_variance\nFROM fact_inventory_audits ia\nJOIN dim_stores s ON ia.store_id = s.store_id\nJOIN dim_product_categories pc ON ia.category_id = pc.category_id\nGROUP BY s.name, pc.name\nORDER BY avg_shrinkage DESC"""
        },
        {
            "question": "Show store incidents by type.",
            "query": """SELECT si.incident_type, si.severity,\n       COUNT(*) AS incidents,\n       AVG(si.resolution_time_min) AS avg_resolution,\n       SUM(CASE WHEN si.reported_to_police = true THEN 1 ELSE 0 END) AS police_reports\nFROM fact_store_incidents si\nGROUP BY si.incident_type, si.severity\nORDER BY incidents DESC"""
        },
        {
            "question": "Show shift handoff completeness.",
            "query": """SELECT sm.name AS from_manager,\n       COUNT(*) AS handoffs,\n       AVG(sh.doc_completeness_pct) AS avg_completeness,\n       SUM(CASE WHEN sh.cash_verified_flag = false THEN 1 ELSE 0 END) AS no_cash_verify\nFROM fact_shift_handoffs sh\nJOIN dim_store_managers sm ON sh.from_manager_id = sm.manager_id\nGROUP BY sm.name\nORDER BY avg_completeness ASC"""
        },
        {
            "question": "Show audit quality by manager.",
            "query": """SELECT sm.name, sm.title,\n       AVG(aq.quality_score) AS avg_quality,\n       AVG(aq.completeness_pct) AS avg_completeness,\n       SUM(aq.errors_found) AS total_errors\nFROM fact_audit_quality aq\nJOIN dim_store_managers sm ON aq.manager_id = sm.manager_id\nGROUP BY sm.name, sm.title\nORDER BY avg_quality DESC"""
        },
        {
            "question": "Show associate coverage by store.",
            "query": """SELECT s.name AS store,\n       AVG(ap.coverage_pct) AS avg_coverage,\n       AVG(ap.associates_scheduled) AS avg_scheduled,\n       AVG(ap.associates_present) AS avg_present\nFROM fact_associate_presence ap\nJOIN dim_stores s ON ap.store_id = s.store_id\nGROUP BY s.name\nORDER BY avg_coverage ASC"""
        },
        {
            "question": "Show scheduling issues.",
            "query": """SELECT s.name AS store, se.event_type, se.shift,\n       SUM(se.unfilled_shifts) AS total_unfilled,\n       SUM(se.headcount) AS total_headcount\nFROM fact_scheduling_events se\nJOIN dim_stores s ON se.store_id = s.store_id\nGROUP BY s.name, se.event_type, se.shift\nORDER BY total_unfilled DESC"""
        },
        {
            "question": "Show manager wellness.",
            "query": """SELECT sm.name, sm.title,\n       mw.workload_score, mw.overtime_hours,\n       mw.stress_level, mw.satisfaction_score, mw.burnout_risk\nFROM fact_manager_wellness mw\nJOIN dim_store_managers sm ON mw.manager_id = sm.manager_id\nORDER BY mw.workload_score DESC"""
        },
        {
            "question": "Show compliance document status.",
            "query": """SELECT cde.doc_type, cde.status,\n       COUNT(*) AS documents,\n       SUM(cde.corrective_actions) AS total_corrective\nFROM fact_compliance_doc_events cde\nGROUP BY cde.doc_type, cde.status\nORDER BY total_corrective DESC"""
        },
    ],

    LAKEHOUSE_NAME: [
        {
            "question": "Which stores have the most POS transactions?",
            "query": """SELECT s.name, s.format,\n       COUNT(*) AS txns,\n       SUM(pt.total_amount) AS total_revenue\nFROM fact_pos_transactions pt\nJOIN dim_stores s ON pt.store_id = s.store_id\nGROUP BY s.name, s.format\nORDER BY txns DESC\nLIMIT 20"""
        },
        {
            "question": "Show stores by district.",
            "query": """SELECT d.name AS district, d.region,\n       COUNT(*) AS stores\nFROM dim_stores s\nJOIN dim_districts d ON s.district_id = d.district_id\nGROUP BY d.name, d.region\nORDER BY stores DESC"""
        },
        {
            "question": "Show product categories.",
            "query": """SELECT department, name, sub_category, margin_pct\nFROM dim_product_categories\nORDER BY department, name"""
        },
        {
            "question": "Show inventory scan summary.",
            "query": """SELECT scan_type, COUNT(*) AS scans,\n       SUM(quantity) AS total_qty\nFROM fact_inventory_scans\nGROUP BY scan_type\nORDER BY scans DESC"""
        },
        {
            "question": "Show vendor partner overview.",
            "query": """SELECT category, COUNT(*) AS vendors,\n       AVG(reliability_score) AS avg_reliability,\n       AVG(lead_time_days) AS avg_lead_time\nFROM dim_vendor_partners\nGROUP BY category\nORDER BY avg_reliability DESC"""
        },
    ],

    EVENTHOUSE_DATABASE: [
        {
            "question": "Show foot traffic by store.",
            "query": """foot_traffic\n| where timestamp > ago(1h)\n| summarize total_count = sum(count) by store_id, zone\n| order by total_count desc"""
        },
        {
            "question": "Show real-time POS transactions.",
            "query": """pos_transactions\n| where timestamp > ago(1h)\n| summarize txns = count(), revenue = sum(total_amount) by store_id\n| order by revenue desc"""
        },
        {
            "question": "Show store incidents.",
            "query": """store_incidents\n| where timestamp > ago(24h)\n| summarize count() by incident_type, severity\n| order by count_ desc"""
        },
        {
            "question": "Show associate location.",
            "query": """associate_location\n| where timestamp > ago(1h)\n| summarize count() by store_id, zone, activity\n| order by count_ desc"""
        },
        {
            "question": "Show inventory scan activity.",
            "query": """inventory_scans\n| where timestamp > ago(24h)\n| summarize count(), total_qty = sum(quantity) by store_id, scan_type\n| order by count_ desc"""
        },
    ],
}

print(f"QA_FEWSHOTS set: {', '.join(f'{k}: {len(v)} queries' for k, v in QA_FEWSHOTS.items())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OPS AGENT — PER-DATASOURCE EXAMPLE QUERIES (fewshots)
# ════════════════════════════════════════════════════════════════════════

OPS_FEWSHOTS = {
    WAREHOUSE_NAME: [
        {
            "question": "Are there critical store incidents?",
            "query": """SELECT s.name AS store, si.incident_type,\n       si.severity, si.resolution_time_min, si.reported_to_police,\n       sm.name AS manager\nFROM fact_store_incidents si\nJOIN dim_stores s ON si.store_id = s.store_id\nJOIN dim_store_managers sm ON si.manager_id = sm.manager_id\nWHERE si.severity = 'Critical'\nORDER BY si.timestamp DESC"""
        },
        {
            "question": "Which stores have high shrinkage?",
            "query": """SELECT s.name AS store, pc.name AS category,\n       ia.shrinkage_pct, ia.variance_dollars,\n       CASE WHEN ia.shrinkage_pct > 10 THEN 'CRITICAL'\n            WHEN ia.shrinkage_pct > 5 THEN 'WARNING'\n            ELSE 'OK' END AS severity\nFROM fact_inventory_audits ia\nJOIN dim_stores s ON ia.store_id = s.store_id\nJOIN dim_product_categories pc ON ia.category_id = pc.category_id\nWHERE ia.shrinkage_pct > 5\nORDER BY ia.shrinkage_pct DESC"""
        },
        {
            "question": "Which stores have low associate coverage?",
            "query": """SELECT s.name AS store, ap.department,\n       ap.associates_scheduled, ap.associates_present, ap.coverage_pct\nFROM fact_associate_presence ap\nJOIN dim_stores s ON ap.store_id = s.store_id\nWHERE ap.coverage_pct < 70\nORDER BY ap.coverage_pct ASC"""
        },
        {
            "question": "Show customer satisfaction drops.",
            "query": """SELECT s.name AS store,\n       cs.overall_score, cs.service_score, cs.checkout_score, cs.nps\nFROM fact_customer_satisfaction cs\nJOIN dim_stores s ON cs.store_id = s.store_id\nWHERE cs.overall_score < 5\nORDER BY cs.overall_score ASC"""
        },
        {
            "question": "Which managers are burned out?",
            "query": """SELECT sm.name, sm.title,\n       mw.workload_score, mw.stress_level,\n       mw.overtime_hours, mw.burnout_risk\nFROM fact_manager_wellness mw\nJOIN dim_store_managers sm ON mw.manager_id = sm.manager_id\nWHERE mw.burnout_risk = true\nORDER BY mw.stress_level DESC"""
        },
        {
            "question": "Show compliance failures.",
            "query": """SELECT s.name AS store, cde.doc_type,\n       cde.status, cde.findings, cde.corrective_actions\nFROM fact_compliance_doc_events cde\nJOIN dim_stores s ON cde.store_id = s.store_id\nWHERE cde.status = 'Failed'\nORDER BY cde.date DESC"""
        },
        {
            "question": "Show scheduling gaps.",
            "query": """SELECT s.name AS store, se.shift,\n       se.unfilled_shifts, se.headcount, se.event_type\nFROM fact_scheduling_events se\nJOIN dim_stores s ON se.store_id = s.store_id\nWHERE se.unfilled_shifts > 3\nORDER BY se.unfilled_shifts DESC"""
        },
        {
            "question": "Show shift handoffs missing cash verification.",
            "query": """SELECT s.name AS store,\n       sm.name AS from_manager,\n       sh.doc_completeness_pct, sh.pending_items\nFROM fact_shift_handoffs sh\nJOIN dim_stores s ON sh.store_id = s.store_id\nJOIN dim_store_managers sm ON sh.from_manager_id = sm.manager_id\nWHERE sh.cash_verified_flag = false\nORDER BY sh.doc_completeness_pct ASC"""
        },
        {
            "question": "Show POS anomalies.",
            "query": """SELECT s.name AS store,\n       pt.total_amount, pt.discount_pct, pt.items, pt.payment_method\nFROM fact_pos_transactions pt\nJOIN dim_stores s ON pt.store_id = s.store_id\nWHERE pt.discount_pct > 50\nORDER BY pt.discount_pct DESC"""
        },
        {
            "question": "Show audit quality issues.",
            "query": """SELECT sm.name AS manager,\n       aq.quality_score, aq.errors_found, aq.revision_count\nFROM fact_audit_quality aq\nJOIN dim_store_managers sm ON aq.manager_id = sm.manager_id\nWHERE aq.quality_score < 70\nORDER BY aq.quality_score ASC"""
        },
    ],

    EVENTHOUSE_DATABASE: [
        {
            "question": "Show critical store incidents.",
            "query": """store_incidents\n| where severity == 'Critical'\n| project store_id, incident_type, severity\n| order by timestamp desc"""
        },
        {
            "question": "Show foot traffic anomalies.",
            "query": """foot_traffic\n| where timestamp > ago(1h)\n| summarize total_count = sum(count) by store_id, zone\n| order by total_count desc"""
        },
        {
            "question": "Show POS transaction anomalies.",
            "query": """pos_transactions\n| where timestamp > ago(24h)\n| where discount_pct > 50\n| project store_id, total_amount, discount_pct, items\n| order by discount_pct desc"""
        },
        {
            "question": "Show inventory scan activity.",
            "query": """inventory_scans\n| where timestamp > ago(24h)\n| summarize count(), total_qty = sum(quantity) by store_id, scan_type\n| order by count_ desc"""
        },
        {
            "question": "Show associate coverage by zone.",
            "query": """associate_location\n| where timestamp > ago(1h)\n| summarize count() by store_id, zone, activity\n| order by count_ desc"""
        },
        {
            "question": "Show POS revenue trend.",
            "query": """pos_transactions\n| where timestamp > ago(7d)\n| summarize revenue = sum(total_amount) by bin(timestamp, 1d), store_id\n| order by timestamp asc"""
        },
        {
            "question": "Show foot traffic trend.",
            "query": """foot_traffic\n| where timestamp > ago(7d)\n| summarize total = sum(count) by bin(timestamp, 1h)\n| order by timestamp asc"""
        },
    ],
}

print(f"OPS_FEWSHOTS set: {', '.join(f'{k}: {len(v)} queries' for k, v in OPS_FEWSHOTS.items())}")

## Sample Questions for the Retail Data Agents

### QA Agent — `Retail_QA_Agent`

| # | Category | Sample Question |
|---|----------|------------------|
| 1 | Customer | What is the customer satisfaction by store? |
| 2 | POS | Show POS transaction summary. |
| 3 | Inventory | Which stores have inventory shrinkage? |
| 4 | Incidents | Show store incidents by type. |
| 5 | Handoffs | Show shift handoff completeness. |
| 6 | Quality | Show audit quality by manager. |
| 7 | Staffing | Show associate coverage by store. |
| 8 | Scheduling | Show scheduling issues. |
| 9 | Wellness | Show manager wellness. |
| 10 | Compliance | Show compliance document status. |
| 11 | Traffic | Show foot traffic by store. |
| 12 | Products | Show product categories. |
| 13 | Vendors | Show vendor partner overview. |
| 14 | Scans | Show inventory scan summary. |
| 15 | Districts | Show stores by district. |

### Ops Agent — `Retail_Ops_Agent`

| # | Category | Sample Question |
|---|----------|------------------|
| 1 | Incidents | Are there critical store incidents? |
| 2 | Inventory | Which stores have high shrinkage? |
| 3 | Staffing | Which stores have low associate coverage? |
| 4 | Customer | Show customer satisfaction drops. |
| 5 | Wellness | Which managers are burned out? |
| 6 | Compliance | Show compliance failures. |
| 7 | Scheduling | Show scheduling gaps. |
| 8 | Handoffs | Show shift handoffs missing cash verification. |
| 9 | POS | Show POS anomalies. |
| 10 | Traffic | Show foot traffic anomalies. |
| 11 | Scans | Show inventory scan activity. |
| 12 | Coverage | Show associate coverage by zone. |
| 13 | Quality | Show audit quality issues. |
| 14 | Trends | Show POS revenue trend. |

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SAVE INSTRUCTIONS FOR PARENT NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

import json

_result = json.dumps({
    "qa": QA_AGENT_INSTRUCTIONS,
    "ops": OPS_AGENT_INSTRUCTIONS,
    "qa_fewshots": QA_FEWSHOTS,
    "ops_fewshots": OPS_FEWSHOTS,
    "qa_ds_instructions": QA_DS_INSTRUCTIONS,
    "ops_ds_instructions": OPS_DS_INSTRUCTIONS
})

_tmp_path = "file:/lakehouse/default/Files/_temp/agent_instructions.json"
notebookutils.fs.put(_tmp_path, _result, True)
print(f"  Written {len(_result):,} bytes to {_tmp_path}")

print(f"{'═'*60}")
print(f"AGENT INSTRUCTIONS LOADED — {INDUSTRY.upper()}")
print(f"{'═'*60}")
print(f"")
print(f"  QA_AGENT_INSTRUCTIONS:  {len(QA_AGENT_INSTRUCTIONS):,} chars")
print(f"  OPS_AGENT_INSTRUCTIONS: {len(OPS_AGENT_INSTRUCTIONS):,} chars")
print(f"  QA_FEWSHOTS:  {sum(len(v) for v in QA_FEWSHOTS.values())} total queries across {len(QA_FEWSHOTS)} datasources")
print(f"  OPS_FEWSHOTS: {sum(len(v) for v in OPS_FEWSHOTS.values())} total queries across {len(OPS_FEWSHOTS)} datasources")

notebookutils.notebook.exit("ok")